In [22]:
from pathlib import Path
import torch
from rfdetr import RFDETRNano

ROOT = Path(r"C:\Users\Administrator\COMP9444_RFDETR")


In [19]:
DATASET_VERSION = "material"

DATASET_DIRS = {
    "instance": ROOT / "trashcan_rfdetr",
    "material": ROOT / "trashcan_material_rfdetr"
}

DATASET_DIR = DATASET_DIRS[DATASET_VERSION]

OUTPUT_DIR = ROOT / "outputs" / f"{DATASET_VERSION}_baseline_notebook"

print("Dataset:", DATASET_VERSION)
print("Dataset directory:", DATASET_DIR)

assert (DATASET_DIR / "train" / "_annotations.coco.json").exists()
assert (DATASET_DIR / "valid" / "_annotations.coco.json").exists()

Dataset: material
Dataset directory: C:\Users\Administrator\COMP9444_RFDETR\trashcan_material_rfdetr


In [20]:
import json
import shutil
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\COMP9444_RFDETR")
SOURCE = ROOT / "trashcan_raw" / "dataset" / "material_version"
TARGET = ROOT / "trashcan_material_rfdetr"

image_files = {}

for p in SOURCE.rglob("*"):
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        if p.name in image_files:
            raise ValueError(f"repeated pic: {p.name}")
        image_files[p.name] = p

print("Total images found:", len(image_files))

splits = {
    "train": "instances_train_trashcan.json",
    "valid": "instances_val_trashcan.json"
}

for split, json_name in splits.items():

    matches = list(SOURCE.rglob(json_name))
    assert len(matches) == 1, f": {json_name}"

    with open(matches[0], encoding="utf-8") as f:
        data = json.load(f)

    folder = TARGET / split
    folder.mkdir(parents=True, exist_ok=True)

    for img in data["images"]:
        name = Path(img["file_name"]).name

        assert name in image_files, f"can't find: {name}"

        destination = folder / name

        if not destination.exists():
            shutil.copy2(image_files[name], destination)

        img["file_name"] = name

    with open(
        folder / "_annotations.coco.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(data, f)

    print(
        split,
        "Images:", len(data["images"]),
        "Annotations:", len(data["annotations"]),
        "Categories:", len(data["categories"])
    )

Total images found: 7212
train Images: 6008 Annotations: 9741 Categories: 16
valid Images: 1204 Annotations: 2595 Categories: 16


In [21]:
import time
import random
import numpy as np
import torch
from rfdetr import RFDETRNano
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\COMP9444_RFDETR")

DATASET_VERSION = "material"

DATASET_DIR = ROOT / "trashcan_material_rfdetr"

OUTPUT_DIR = (
    ROOT / "outputs" / "material_baseline_notebook_10"
)

EPOCHS = 10
BATCH_SIZE = 4
LEARNING_RATE = 1e-4


random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


assert torch.cuda.is_available()

assert (
    DATASET_DIR / "train" / "_annotations.coco.json"
).exists()

assert (
    DATASET_DIR / "valid" / "_annotations.coco.json"
).exists()


model = RFDETRNano()

start = time.perf_counter()

model.train(
    dataset_dir=str(DATASET_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=1,
    lr=LEARNING_RATE,
    num_workers=0,
    device="cuda",
    progress_bar="tqdm",
    compute_val_loss=True,
    output_dir=str(OUTPUT_DIR)
)


elapsed = time.perf_counter() - start

print(f"Elapsed minutes: {elapsed / 60:.2f}")

Val (ema) (Epoch 10/10) — Overall Metrics            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.4232 │ 0.6268 │ 0.4642 │ 0.6561 │ 0.6343 │ 0.7616 │ 0.5537 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘


Epoch 9: 100%|██████████| 1502/1502 [06:40<00:00,  3.75it/s, loss=4.970, loss_cls=0.819, loss_box=0.0734, loss_giou=0.210, train/lr=0.0001, val/loss=4.260, val/ema_mAP_50_95=0.423, val/mAP_50_95=0.423, val/F1=0.634, train/loss=3.460, max_mem=3.6 GB, free_mem=2.9 GB/8.0 GB]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 1502/1502 [06:46<00:00,  3.69it/s, loss=4.970, loss_cls=0.819, loss_box=0.0734, loss_giou=0.210, train/lr=0.0001, val/loss=4.260, val/ema_mAP_50_95=0.423, val/mAP_50_95=0.423, val/F1=0.634, train/loss=3.460, max_mem=3.6 GB, free_mem=2.9 GB/8.0 GB]
[2026-11-07 12:49:30] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.0000, ema=0.4232)
Elapsed minutes: 67.31
